# Thailand Digital CPI: Lazada Price Tracker

## An e-commerce price pressure analysis of Thailand’s online health-products market

Official Consumer Price Index data is useful, but it is usually released after prices have already changed in the real economy. This project explores whether e-commerce product listings can act as a faster digital signal of price pressure in Thailand.

Using Lazada Thailand product-listing data, this notebook builds a mock **Digital CPI Dashboard** by analyzing product prices, discounts, seller concentration, and product tiers.

The goal is not to replace the official CPI. Instead, this project asks whether online marketplace data can reveal early signs of inflation pressure and premium markups in Thailand’s digital retail economy.

### Research Question

**To what extent can Lazada Thailand product-listing data reveal digital price pressure, and premium markups across online consumer-health categories?**

### Project Summary

This notebook creates a mock Digital CPI Dashboard for Thailand’s e-commerce market using Lazada product-listing data via combining economic theory, data cleaning, simple NLP classification, discount-depth analysis, and interactive visualization to analyze online price pressure.

In [1]:
import os
import re
import glob
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Dataset and Research Design

The dataset used in this project is a Lazada Thailand health-products dataset. It contains product listings from Thailand’s online marketplace ecosystem.

This project uses a cross-sectional design. That means it studies price differences across products, sellers, and categories at one point in time. Because the dataset is not a full monthly time series, this notebook does not calculate true inflation over time. Instead, it creates a mock digital CPI-style dashboard based on current online price pressure.

### Key Variables

The project attempts to identify:

- Product title
- Current price
- Regular/original price
- Seller or shop name
- Product category
- Number sold or review count
- Rating
- Location

### Main Metrics

- **Discount Depth**: how large the discount is compared to the original price
- **Premium Markup**: how much more premium products cost compared to necessity products
- **Seller Market Share**: seller concentration using sales or listing counts
- **Digital Price Pressure Index**: category median price compared with the overall marketplace median

In [2]:
# Find all CSV files inside Kaggle input folder
csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)

print("CSV files found:")
for file in csv_files:
    print(file)

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV files found. Make sure you added the Lazada Thailand dataset using Add Data.")

# Pick the largest CSV file because it is usually the main dataset
file_sizes = {file: os.path.getsize(file) for file in csv_files}
main_file = max(file_sizes, key=file_sizes.get)

print("\nMain file selected:")
print(main_file)

# Try reading the file with multiple possible encodings
encodings = ["utf-8", "utf-8-sig", "latin1"]

for enc in encodings:
    try:
        df = pd.read_csv(main_file, encoding=enc, on_bad_lines="skip")
        print(f"\nLoaded successfully using encoding: {enc}")
        break
    except Exception as e:
        print(f"Failed with encoding {enc}: {e}")

print("\nDataset shape:", df.shape)
df.head()

CSV files found:
/kaggle/input/datasets/wuttipats/lazada-thailand-health-products-dataset/health_and_wellness_cleaned_20230810_213000.csv

Main file selected:
/kaggle/input/datasets/wuttipats/lazada-thailand-health-products-dataset/health_and_wellness_cleaned_20230810_213000.csv

Loaded successfully using encoding: utf-8

Dataset shape: (68499, 7)


,Id,Section,Name,Price,Total Sold,Total Reviews,Shop Location
0,4799179886,Acne Care,DHC Vitamin B-Mix วิตามินบีรวม (สำหรับ 20 วัน),฿75.00,NaN,NaN,Pathum Thani
1,4795171651,Acne Care,ซิงค์ Vistra Zinc วิสทร้า ซิงค์ 15 มก. ขนาด 20...,฿79.00,NaN,NaN,Chiang Mai
2,4790242641,Acne Care,Blackmores แบลคมอร์ส Bio Zinc A Chelate (90 T...,฿224.00,NaN,NaN,Surin
3,4789940606,Acne Care,1แถม1 กลูต้าวิตมี กลูต้าส้มเลือด Gluta With Me...,฿290.00,NaN,NaN,Udon Thani
4,4787161067,Acne Care,🎌 DHC Vitamin B-Mix Persistent วิตามินบีรวม แบ...,฿149.00,NaN,NaN,Bangkok


In [3]:
print("Column names in this dataset:\n")

for i, col in enumerate(df.columns):
    print(i, ":", col)

print("\nBasic dataset preview:")
display(df.head())

print("\nMissing values preview:")
display(df.isna().sum().sort_values(ascending=False).head(20))

Column names in this dataset:

0 : Id
1 : Section
2 : Name
3 : Price
4 : Total Sold
5 : Total Reviews
6 : Shop Location

Basic dataset preview:


,Id,Section,Name,Price,Total Sold,Total Reviews,Shop Location
0,4799179886,Acne Care,DHC Vitamin B-Mix วิตามินบีรวม (สำหรับ 20 วัน),฿75.00,NaN,NaN,Pathum Thani
1,4795171651,Acne Care,ซิงค์ Vistra Zinc วิสทร้า ซิงค์ 15 มก. ขนาด 20...,฿79.00,NaN,NaN,Chiang Mai
2,4790242641,Acne Care,Blackmores แบลคมอร์ส Bio Zinc A Chelate (90 T...,฿224.00,NaN,NaN,Surin
3,4789940606,Acne Care,1แถม1 กลูต้าวิตมี กลูต้าส้มเลือด Gluta With Me...,฿290.00,NaN,NaN,Udon Thani
4,4787161067,Acne Care,🎌 DHC Vitamin B-Mix Persistent วิตามินบีรวม แบ...,฿149.00,NaN,NaN,Bangkok



Missing values preview:


Total Sold       31797
Total Reviews    27547
Shop Location       69
Name                 0
Section              0
Id                   0
Price                0
dtype: int64

In [4]:
def find_column(columns, include_keywords, exclude_keywords=None):
    """
    Finds the first column whose name contains one of the include keywords
    and does not contain excluded keywords.
    """
    exclude_keywords = exclude_keywords or []
    
    for col in columns:
        col_lower = col.lower()
        include_match = any(keyword in col_lower for keyword in include_keywords)
        exclude_match = any(keyword in col_lower for keyword in exclude_keywords)
        
        if include_match and not exclude_match:
            return col
    
    return None


columns = list(df.columns)

title_col = find_column(columns, ["title", "name", "product"])
description_col = find_column(columns, ["description", "desc", "detail"])
category_col = find_column(columns, ["category", "cat", "type"])
seller_col = find_column(columns, ["seller", "shop", "merchant", "store"])
location_col = find_column(columns, ["location", "province", "city", "area"])

# Price columns
current_price_col = find_column(
    columns,
    ["sale_price", "discount_price", "discounted_price", "current_price", "price"],
    ["original", "regular", "retail", "old", "before", "mrp", "discount_percent"]
)

regular_price_col = find_column(
    columns,
    ["original", "regular", "retail", "old", "before", "mrp", "list_price"],
    ["percent"]
)

discount_percent_col = find_column(columns, ["discount_percent", "discount percentage", "discount_rate"])
sold_col = find_column(columns, ["sold", "sales", "quantity", "unit"])
rating_col = find_column(columns, ["rating", "stars", "score"])
review_col = find_column(columns, ["review", "reviews", "comment"])

detected_columns = {
    "title_col": title_col,
    "description_col": description_col,
    "category_col": category_col,
    "seller_col": seller_col,
    "location_col": location_col,
    "current_price_col": current_price_col,
    "regular_price_col": regular_price_col,
    "discount_percent_col": discount_percent_col,
    "sold_col": sold_col,
    "rating_col": rating_col,
    "review_col": review_col
}

detected_columns

{'title_col': 'Name',
 'description_col': None,
 'category_col': 'Shop Location',
 'seller_col': 'Shop Location',
 'location_col': 'Shop Location',
 'current_price_col': 'Price',
 'regular_price_col': 'Total Sold',
 'discount_percent_col': None,
 'sold_col': 'Total Sold',
 'rating_col': None,
 'review_col': 'Total Reviews'}

In [5]:
def clean_price(value):
    """
    Converts messy price strings into numeric Thai Baht values.
    Handles symbols like ฿, THB, commas, and price ranges.
    """
    if pd.isna(value):
        return np.nan
    
    value = str(value).lower().strip()
    
    # Remove common currency words and symbols
    value = value.replace("฿", "")
    value = value.replace("thb", "")
    value = value.replace("baht", "")
    value = value.replace(",", "")
    value = value.replace(" ", "")
    
    # Detect numbers
    numbers = re.findall(r"\d+(?:\.\d+)?", value)
    
    if len(numbers) == 0:
        return np.nan
    
    numbers = [float(num) for num in numbers]
    
    # If the price is written as a range, use the midpoint
    if len(numbers) >= 2:
        return np.mean(numbers[:2])
    
    price = numbers[0]
    
    # Handle 1.2k style prices
    if "k" in value:
        price *= 1000
    
    return price


clean_df = df.copy()

if current_price_col is None:
    raise ValueError("Could not find a current price column. Check the printed column names above.")

clean_df["current_price_baht"] = clean_df[current_price_col].apply(clean_price)

if regular_price_col is not None:
    clean_df["regular_price_baht"] = clean_df[regular_price_col].apply(clean_price)
else:
    clean_df["regular_price_baht"] = np.nan

# Remove impossible or extreme price values
clean_df = clean_df[
    (clean_df["current_price_baht"].notna()) &
    (clean_df["current_price_baht"] > 0) &
    (clean_df["current_price_baht"] < clean_df["current_price_baht"].quantile(0.995))
]

print("Cleaned dataset shape:", clean_df.shape)
display(clean_df[["current_price_baht", "regular_price_baht"]].describe())

Cleaned dataset shape: (67977, 9)


,current_price_baht,regular_price_baht
count,67977.000000,36640.000000
mean,481.297563,1551.455731
std,544.430051,7047.394452
min,9.000000,5.000000
25%,159.000000,24.000000
50%,290.000000,128.500000
75%,590.000000,826.000000
max,3800.000000,100000.000000


In [6]:
# Create clean product title
if title_col is not None:
    clean_df["product_title"] = clean_df[title_col].astype(str)
else:
    clean_df["product_title"] = "Unknown Product"

# Create description length
if description_col is not None:
    clean_df["description_text"] = clean_df[description_col].astype(str)
else:
    clean_df["description_text"] = ""

clean_df["title_lower"] = clean_df["product_title"].str.lower()
clean_df["description_length"] = clean_df["description_text"].str.len()

# Create seller variable
if seller_col is not None:
    clean_df["seller_name"] = clean_df[seller_col].astype(str)
else:
    clean_df["seller_name"] = "Unknown Seller"

# Create category variable
if category_col is not None:
    clean_df["category_raw"] = clean_df[category_col].astype(str)
else:
    clean_df["category_raw"] = "Unknown Category"

# Clean category labels
clean_df["category_clean"] = (
    clean_df["category_raw"]
    .str.replace("_", " ")
    .str.replace("-", " ")
    .str.strip()
    .str.title()
)

clean_df[["product_title", "current_price_baht", "regular_price_baht", "seller_name", "category_clean"]].head()

,product_title,current_price_baht,regular_price_baht,seller_name,category_clean
0,DHC Vitamin B-Mix วิตามินบีรวม (สำหรับ 20 วัน),75.0,NaN,Pathum Thani,Pathum Thani
1,ซิงค์ Vistra Zinc วิสทร้า ซิงค์ 15 มก. ขนาด 20...,79.0,NaN,Chiang Mai,Chiang Mai
2,Blackmores แบลคมอร์ส Bio Zinc A Chelate (90 T...,224.0,NaN,Surin,Surin
3,1แถม1 กลูต้าวิตมี กลูต้าส้มเลือด Gluta With Me...,290.0,NaN,Udon Thani,Udon Thani
4,🎌 DHC Vitamin B-Mix Persistent วิตามินบีรวม แบ...,149.0,NaN,Bangkok,Bangkok


In [7]:
def infer_sector(title):
    title = str(title).lower()
    
    if any(word in title for word in ["vitamin", "supplement", "collagen", "zinc", "calcium", "omega", "protein", "วิตามิน"]):
        return "Vitamins & Supplements"
    
    elif any(word in title for word in ["mask", "thermometer", "test", "medical", "glove", "alcohol", "sanitizer", "หน้ากาก"]):
        return "Medical Supplies"
    
    elif any(word in title for word in ["soap", "shampoo", "toothpaste", "toothbrush", "deodorant", "cleanser", "สบู่", "แชมพู"]):
        return "Personal Care"
    
    elif any(word in title for word in ["serum", "cream", "whitening", "skincare", "lotion", "moisturizer", "beauty", "เซรั่ม"]):
        return "Beauty & Skincare"
    
    elif any(word in title for word in ["baby", "diaper", "milk", "เด็ก", "ผ้าอ้อม"]):
        return "Baby & Family"
    
    elif any(word in title for word in ["food", "drink", "tea", "coffee", "snack", "organic", "อาหาร"]):
        return "Food & Nutrition"
    
    else:
        return "Other Health Products"


clean_df["product_sector"] = clean_df["product_title"].apply(infer_sector)

clean_df["product_sector"].value_counts()

product_sector
Other Health Products     38903
Vitamins & Supplements    16824
Food & Nutrition          11500
Baby & Family               466
Beauty & Skincare           176
Personal Care                70
Medical Supplies             38
Name: count, dtype: int64

In [8]:
essential_keywords = [
    "mask", "soap", "shampoo", "toothpaste", "toothbrush", "sanitizer",
    "alcohol", "medical", "thermometer", "diaper", "milk", "baby",
    "วิตามิน", "สบู่", "แชมพู", "หน้ากาก", "ผ้าอ้อม", "นม"
]

premium_keywords = [
    "premium", "luxury", "organic", "imported", "import", "official",
    "collagen", "anti-aging", "whitening", "serum", "gold", "professional",
    "exclusive", "limited", "advanced", "พรีเมียม", "นำเข้า", "ออร์แกนิค",
    "คอลลาเจน", "เซรั่ม"
]


def keyword_count(text, keywords):
    text = str(text).lower()
    return sum(keyword in text for keyword in keywords)


clean_df["essential_keyword_score"] = clean_df["title_lower"].apply(lambda x: keyword_count(x, essential_keywords))
clean_df["premium_keyword_score"] = clean_df["title_lower"].apply(lambda x: keyword_count(x, premium_keywords))

# Use description length as a rough proxy for more premium marketing-heavy listings
description_threshold = clean_df["description_length"].median()

def classify_tier(row):
    if row["premium_keyword_score"] > 0 or row["description_length"] > description_threshold * 1.5:
        return "Luxury/Premium"
    elif row["essential_keyword_score"] > 0:
        return "Essential/Necessity"
    else:
        return "Mass Market/Other"

clean_df["product_tier"] = clean_df.apply(classify_tier, axis=1)

clean_df["product_tier"].value_counts()

product_tier
Mass Market/Other      49945
Luxury/Premium          9060
Essential/Necessity     8972
Name: count, dtype: int64

In [9]:
sample_text = clean_df["product_title"].dropna().astype(str).sample(
    min(10000, len(clean_df)), 
    random_state=42
)

vectorizer = TfidfVectorizer(
    max_features=30,
    stop_words="english",
    token_pattern=r"(?u)\b\w+\b"
)

tfidf_matrix = vectorizer.fit_transform(sample_text)
tfidf_terms = vectorizer.get_feature_names_out()

tfidf_scores = np.asarray(tfidf_matrix.mean(axis=0)).ravel()

tfidf_df = pd.DataFrame({
    "term": tfidf_terms,
    "average_tfidf_score": tfidf_scores
}).sort_values("average_tfidf_score", ascending=False)

tfidf_df.head(20)

,term,average_tfidf_score
16,น,0.147147
18,ม,0.112273
21,ส,0.103699
12,ด,0.098769
19,ล,0.090971
3,ก,0.088476
20,ว,0.086357
0,1,0.076323
8,ง,0.075458
5,กล,0.068417


In [10]:
# Calculate discount depth
clean_df["discount_depth_baht"] = clean_df["regular_price_baht"] - clean_df["current_price_baht"]

clean_df["discount_depth_pct"] = (
    clean_df["discount_depth_baht"] / clean_df["regular_price_baht"]
) * 100

# Keep only sensible discounts
clean_df.loc[clean_df["discount_depth_baht"] < 0, "discount_depth_baht"] = np.nan
clean_df.loc[clean_df["discount_depth_pct"] < 0, "discount_depth_pct"] = np.nan
clean_df.loc[clean_df["discount_depth_pct"] > 95, "discount_depth_pct"] = np.nan

discount_summary = (
    clean_df
    .groupby("product_sector")
    .agg(
        listings=("product_title", "count"),
        median_current_price=("current_price_baht", "median"),
        median_regular_price=("regular_price_baht", "median"),
        median_discount_baht=("discount_depth_baht", "median"),
        median_discount_pct=("discount_depth_pct", "median")
    )
    .reset_index()
    .sort_values("median_discount_pct", ascending=False)
)

discount_summary

,product_sector,listings,median_current_price,median_regular_price,median_discount_baht,median_discount_pct
2,Food & Nutrition,11500,300.0,283.0,679.0,71.895425
6,Vitamins & Supplements,16824,289.0,140.0,832.0,70.472441
0,Baby & Family,466,131.0,27.0,518.0,67.447917
4,Other Health Products,38903,290.0,110.0,697.0,64.301197
1,Beauty & Skincare,176,350.0,53.0,32.0,9.667674
3,Medical Supplies,38,679.5,252.5,NaN,NaN
5,Personal Care,70,3240.0,NaN,NaN,NaN


In [11]:
def clean_number(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value).lower().replace(",", "").strip()
    
    numbers = re.findall(r"\d+(?:\.\d+)?", value)
    
    if len(numbers) == 0:
        return np.nan
    
    number = float(numbers[0])
    
    if "k" in value:
        number *= 1000
    
    return number


if sold_col is not None:
    clean_df["sales_proxy"] = clean_df[sold_col].apply(clean_number)
elif review_col is not None:
    clean_df["sales_proxy"] = clean_df[review_col].apply(clean_number)
else:
    clean_df["sales_proxy"] = 1

clean_df["sales_proxy"] = clean_df["sales_proxy"].fillna(1)

seller_share = (
    clean_df
    .groupby("seller_name")
    .agg(
        listings=("product_title", "count"),
        sales_proxy=("sales_proxy", "sum"),
        median_price=("current_price_baht", "median"),
        median_discount_pct=("discount_depth_pct", "median")
    )
    .reset_index()
)

seller_share["market_share_pct"] = (
    seller_share["sales_proxy"] / seller_share["sales_proxy"].sum()
) * 100

seller_share = seller_share.sort_values("market_share_pct", ascending=False)

seller_share.head(15)

,seller_name,listings,sales_proxy,median_price,median_discount_pct,market_share_pct
2,Bangkok,30015,24353317.0,259.0,67.307692,42.817758
32,Nonthaburi,5326,9385052.0,299.0,67.532468,16.500704
16,Khon Kaen,829,7367299.0,249.0,61.074919,12.953111
33,Pathum Thani,5767,4243304.0,240.0,78.333333,7.460535
51,Samut Prakan,3951,4186456.0,390.0,54.858454,7.360585
50,Sakon Nakhon,389,1146162.0,139.0,65.533063,2.015171
65,Udon Thani,691,895491.0,178.0,74.426808,1.574443
34,Pattani,370,852116.0,155.0,78.886756,1.498182
53,Saraburi,781,486559.0,275.0,80.997877,0.855463
5,Chachoengsao,624,479547.0,340.0,58.227848,0.843135


In [12]:
overall_median_price = clean_df["current_price_baht"].median()

category_dashboard = (
    clean_df
    .groupby("product_sector")
    .agg(
        listings=("product_title", "count"),
        median_price_baht=("current_price_baht", "median"),
        average_price_baht=("current_price_baht", "mean"),
        median_discount_pct=("discount_depth_pct", "median"),
        sales_proxy=("sales_proxy", "sum")
    )
    .reset_index()
)

category_dashboard["listing_share_pct"] = (
    category_dashboard["listings"] / category_dashboard["listings"].sum()
) * 100

category_dashboard["sales_proxy_share_pct"] = (
    category_dashboard["sales_proxy"] / category_dashboard["sales_proxy"].sum()
) * 100

category_dashboard["digital_price_pressure_index"] = (
    category_dashboard["median_price_baht"] / overall_median_price
) * 100

category_dashboard = category_dashboard.sort_values(
    "digital_price_pressure_index", 
    ascending=False
)

category_dashboard

,product_sector,listings,median_price_baht,average_price_baht,median_discount_pct,sales_proxy,listing_share_pct,sales_proxy_share_pct,digital_price_pressure_index
5,Personal Care,70,3240.0,3240.000000,NaN,70.0,0.102976,0.000123,1117.241379
3,Medical Supplies,38,679.5,679.500000,NaN,9595.0,0.055901,0.016870,234.310345
1,Beauty & Skincare,176,350.0,462.977273,9.667674,17471.0,0.258911,0.030717,120.689655
2,Food & Nutrition,11500,300.0,477.845716,71.895425,7953784.0,16.917487,13.984263,103.448276
4,Other Health Products,38903,290.0,488.323767,64.301197,25309174.0,57.229651,44.498336,100.000000
6,Vitamins & Supplements,16824,289.0,461.887137,70.472441,23269528.0,24.749548,40.912251,99.655172
0,Baby & Family,466,131.0,257.049356,67.447917,317053.0,0.685526,0.557439,45.172414


In [13]:
tier_summary = (
    clean_df
    .groupby(["product_sector", "product_tier"])
    .agg(
        listings=("product_title", "count"),
        median_price_baht=("current_price_baht", "median"),
        median_discount_pct=("discount_depth_pct", "median")
    )
    .reset_index()
)

tier_pivot = tier_summary.pivot_table(
    index="product_sector",
    columns="product_tier",
    values="median_price_baht"
).reset_index()

if "Luxury/Premium" in tier_pivot.columns and "Essential/Necessity" in tier_pivot.columns:
    tier_pivot["premium_markup_pct"] = (
        (tier_pivot["Luxury/Premium"] - tier_pivot["Essential/Necessity"]) /
        tier_pivot["Essential/Necessity"]
    ) * 100
else:
    tier_pivot["premium_markup_pct"] = np.nan

tier_pivot.sort_values("premium_markup_pct", ascending=False)

product_tier,product_sector,Essential/Necessity,Luxury/Premium,Mass Market/Other,premium_markup_pct
0,Baby & Family,290.0,899.0,131.0,210.000000
6,Vitamins & Supplements,235.0,295.0,395.0,25.531915
4,Other Health Products,330.0,390.0,285.0,18.181818
2,Food & Nutrition,509.0,290.0,299.0,-43.025540
1,Beauty & Skincare,NaN,299.0,350.0,NaN
3,Medical Supplies,NaN,NaN,679.5,NaN
5,Personal Care,3240.0,NaN,NaN,NaN


In [14]:
fig = px.box(
    clean_df,
    x="product_tier",
    y="current_price_baht",
    points=False,
    title="Price Distribution by Product Tier",
    labels={
        "product_tier": "Product Tier",
        "current_price_baht": "Current Price (Thai Baht)"
    }
)

fig.update_yaxes(type="log")
fig.show()

In [15]:
discount_plot = discount_summary.dropna(subset=["median_discount_pct"]).copy()

fig = px.bar(
    discount_plot,
    x="product_sector",
    y="median_discount_pct",
    title="Median Discount Depth by Product Sector",
    labels={
        "product_sector": "Product Sector",
        "median_discount_pct": "Median Discount Depth (%)"
    }
)

fig.update_layout(xaxis_tickangle=-35)
fig.show()

In [16]:
fig = px.bar(
    category_dashboard,
    x="product_sector",
    y="digital_price_pressure_index",
    title="Mock Digital CPI: Price Pressure Index by Sector",
    labels={
        "product_sector": "Product Sector",
        "digital_price_pressure_index": "Digital Price Pressure Index"
    }
)

fig.add_hline(
    y=100,
    line_dash="dash",
    annotation_text="Marketplace Median = 100",
    annotation_position="top left"
)

fig.update_layout(xaxis_tickangle=-35)
fig.show()

In [17]:
top_sellers = seller_share.head(15).copy()

fig = px.bar(
    top_sellers,
    x="seller_name",
    y="market_share_pct",
    title="Top Sellers by Estimated Market Share",
    labels={
        "seller_name": "Seller",
        "market_share_pct": "Estimated Market Share (%)"
    }
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [18]:
sector_strategy = category_dashboard.merge(
    discount_summary[["product_sector", "median_discount_pct"]],
    on="product_sector",
    how="left",
    suffixes=("", "_discount")
)

fig = px.scatter(
    sector_strategy,
    x="median_price_baht",
    y="median_discount_pct_discount",
    size="listing_share_pct",
    hover_name="product_sector",
    title="Sector Pricing Strategy: Price Level vs Discount Depth",
    labels={
        "median_price_baht": "Median Price (Thai Baht)",
        "median_discount_pct_discount": "Median Discount Depth (%)",
        "listing_share_pct": "Listing Share (%)"
    }
)

fig.show()

In [19]:
dashboard_data = category_dashboard.merge(
    discount_summary[["product_sector", "median_discount_pct"]],
    on="product_sector",
    how="left",
    suffixes=("", "_discount")
)

# Fix: Pie charts need subplot type "domain"
fig = make_subplots(
    rows=2,
    cols=2,
    specs=[
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "domain"}, {"type": "xy"}]
    ],
    subplot_titles=(
        "Digital Price Pressure Index",
        "Median Discount Depth",
        "Listing Share by Sector",
        "Median Price by Sector"
    )
)

fig.add_trace(
    go.Bar(
        x=dashboard_data["product_sector"],
        y=dashboard_data["digital_price_pressure_index"],
        name="Price Pressure Index"
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Bar(
        x=dashboard_data["product_sector"],
        y=dashboard_data["median_discount_pct_discount"],
        name="Discount Depth"
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Pie(
        labels=dashboard_data["product_sector"],
        values=dashboard_data["listing_share_pct"],
        name="Listing Share",
        hole=0.35
    ),
    row=2,
    col=1
)

fig.add_trace(
    go.Bar(
        x=dashboard_data["product_sector"],
        y=dashboard_data["median_price_baht"],
        name="Median Price"
    ),
    row=2,
    col=2
)

fig.update_layout(
    title_text="Thailand E-Commerce Digital CPI Dashboard",
    height=900,
    showlegend=False
)

fig.update_xaxes(tickangle=-35, row=1, col=1)
fig.update_xaxes(tickangle=-35, row=1, col=2)
fig.update_xaxes(tickangle=-35, row=2, col=2)

fig.show()

In [20]:
highest_price_pressure = category_dashboard.iloc[0]
lowest_price_pressure = category_dashboard.iloc[-1]

highest_discount = discount_summary.dropna(subset=["median_discount_pct"]).iloc[0]

top_seller = seller_share.iloc[0]

print("KEY FINDINGS")
print("------------")
print(f"1. Highest digital price pressure sector: {highest_price_pressure['product_sector']}")
print(f"   Digital Price Pressure Index: {highest_price_pressure['digital_price_pressure_index']:.2f}")

print(f"\n2. Lowest digital price pressure sector: {lowest_price_pressure['product_sector']}")
print(f"   Digital Price Pressure Index: {lowest_price_pressure['digital_price_pressure_index']:.2f}")

print(f"\n3. Sector with deepest median discounts: {highest_discount['product_sector']}")
print(f"   Median Discount Depth: {highest_discount['median_discount_pct']:.2f}%")

print(f"\n4. Largest seller by estimated market share: {top_seller['seller_name']}")
print(f"   Estimated Market Share: {top_seller['market_share_pct']:.2f}%")

print(f"\n5. Overall marketplace median price: ฿{overall_median_price:.2f}")

KEY FINDINGS
------------
1. Highest digital price pressure sector: Personal Care
   Digital Price Pressure Index: 1117.24

2. Lowest digital price pressure sector: Baby & Family
   Digital Price Pressure Index: 45.17

3. Sector with deepest median discounts: Food & Nutrition
   Median Discount Depth: 71.90%

4. Largest seller by estimated market share: Bangkok
   Estimated Market Share: 42.82%

5. Overall marketplace median price: ฿290.00


## Economic Interpretation

The results suggest that Thailand’s e-commerce marketplace does not behave as one uniform consumer market; rather, different product sectors show different levels of digital price pressure.

Sectors with a Digital Price Pressure Index above 100 are priced above the marketplace median. These sectors may reflect stronger brand power, higher input costs, imported-product exposure, or premium positioning. Sectors below 100 appear more price competitive and may contain more basic necessity goods.

Discount depth provides another signal. A high median discount does not automatically mean consumers are better off. It may also suggest that sellers are cutting margins aggressively to maintain demand in a competitive or slowing market. In this way, discounting becomes a digital signal of market stress.

The Essential/Necessity vs Luxury/Premium classification adds an economic layer to the analysis. If premium products show much higher median prices than necessity goods, this may suggest that online marketplaces contain strong product stratification, where higher-income consumers face a very different price environment than lower-income consumers.

## Limitations

1. The dataset is a cross-sectional product-listing snapshot. It does not track the same products month by month, so it cannot calculate true inflation over time.

2. Online prices may not represent all Thai consumers. Lazada users may differ from consumers who shop mainly in physical stores, wet markets, or local pharmacies.

3. Discounts on e-commerce platforms can be strategic. A large discount may reflect marketing behavior rather than a real fall in production costs.

4. The Essential/Necessity vs Luxury/Premium classification uses simple keyword-based NLP. This is transparent and beginner-friendly, but it is less accurate than a supervised machine-learning classifier trained on manually labeled products.

5. Seller market share is estimated using available variables such as sales, reviews, or listing count. If exact transaction data is unavailable, this should be treated as a proxy rather than a perfect measure of market share.

## Conclusion

This project built a mock Digital CPI Dashboard using Lazada Thailand e-commerce product data. By cleaning Thai Baht price variables, classifying products using NLP, calculating discount depth, and estimating seller/category market share, the notebook shows how online marketplace data can reveal digital price pressure.

The strongest insight is that e-commerce prices are not only about inflation. They also reflect seller competition, discount strategy, product positioning, and consumer segmentation. This makes e-commerce data valuable for modern economic analysis, especially in countries like Thailand where digital retail is increasingly important.

The next step would be to repeat this analysis across multiple monthly snapshots from Lazada or Shopee. That would allow the project to move from a cross-sectional price pressure tracker toward a true digital inflation index.